# Cross-Validation, Model Selection, and External Test Evaluation

This notebook demonstrates how to:
- Use cross-validation for robust model selection
- Compare model performance with internal and external test sets
- Save the best model
- Make predictions on new (external) data
- Apply statistical inference to compare models

**Audience:** This is designed for learners who want to understand best practices in model evaluation and deployment.

## 1. Import Required Libraries
We import all necessary libraries for data handling, modeling, cross-validation, and plotting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from scipy import stats
import joblib
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)


## 2. Load and Explore the Dataset
We use the UCI Heart Disease dataset.

In [ ]:
from ucimlrepo import fetch_ucirepo
hd = fetch_ucirepo(id=45)
df = hd.data.features.copy()
df['target'] = (hd.data.targets['num'] > 0).astype(int)
print('Shape:', df.shape)
display(df.head())
sns.countplot(x='target', data=df)
plt.title('Class Distribution')
plt.show()

## 3. Preprocess the Data
Impute missing values, encode categoricals, and scale features.

In [ ]:
# Impute missing values (median for numeric)
for col in df.select_dtypes('number').columns:
    df[col].fillna(df[col].median(), inplace=True)

# Encode categoricals if any
for col in df.select_dtypes('object').columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Split features/target
X = df.drop('target', axis=1).values
y = df['target'].values

# Remove any rows with NaNs (robustness)
mask = ~np.isnan(X).any(axis=1)
X = X[mask]
y = y[mask]

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)


## 4. Split Data: Internal Train/Test and External Test
We simulate a real-world scenario by holding out an external test set before any model selection.

In [ ]:
# Hold out 20% as external test set (never seen during model selection)
X_dev, X_ext, y_dev, y_ext = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Development set:', X_dev.shape, 'External test set:', X_ext.shape)

# Internal split for cross-validation
print('Class balance in external test:', np.bincount(y_ext))

## 5. Cross-Validation for Model Selection
We use Stratified K-Fold cross-validation on the development set to compare models.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_dev, y_dev, cv=cv, scoring='roc_auc')
    cv_results[name] = scores
    print(f'{name}: mean ROC-AUC={scores.mean():.3f} ± {scores.std():.3f}')

# Boxplot for visual comparison
cv_df = pd.DataFrame(cv_results)
sns.boxplot(data=cv_df)
plt.title('Cross-Validated ROC-AUC Scores')
plt.ylabel('ROC-AUC')
plt.show()

## 6. Statistical Comparison of Models
Apply paired t-tests to compare cross-validated scores.

In [ ]:
from itertools import combinations
print('Paired t-tests for ROC-AUC:')
for m1, m2 in combinations(cv_df.columns, 2):
    t_stat, p_val = stats.ttest_rel(cv_df[m1], cv_df[m2])
    print(f'{m1} vs {m2}: t={t_stat:.3f}, p={p_val:.3g}')

## 7. Retrain Best Model on All Development Data
Select the best model and fit it to the entire development set.

In [ ]:
# Identify best model by mean ROC-AUC
best_model_name = cv_df.mean().idxmax()
print(f'Best model: {best_model_name}')
best_model = models[best_model_name]
best_model.fit(X_dev, y_dev)

## 8. Save the Best Model
Persist the trained model to disk for future use.

In [ ]:
joblib.dump(best_model, f'best_model_{best_model_name.replace(" ", "_").lower()}.joblib')
print('Model saved!')

## 9. Predict and Evaluate on External Test Set
Load the saved model and evaluate its performance on the external test set.

In [ ]:
# Load model (simulate deployment)
loaded_model = joblib.load(f'best_model_{best_model_name.replace(" ", "_").lower()}.joblib')
y_pred = loaded_model.predict(X_ext)
y_prob = loaded_model.predict_proba(X_ext)[:, 1] if hasattr(loaded_model, 'predict_proba') else loaded_model.decision_function(X_ext)

acc = accuracy_score(y_ext, y_pred)
f1 = f1_score(y_ext, y_pred)
roc = roc_auc_score(y_ext, y_prob)
print(f'External Test — Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}')
print(classification_report(y_ext, y_pred))


## 10. Statistical Inference: Internal vs External Performance
Compare cross-validated and external test ROC-AUC for the best model.

In [ ]:
# Compare mean CV ROC-AUC to external ROC-AUC
cv_auc = cv_df[best_model_name]
ext_auc = roc
print(f'Best model ({best_model_name}) — CV ROC-AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}, External ROC-AUC: {ext_auc:.3f}')
plt.figure(figsize=(6,4))
sns.boxplot(data=cv_auc, orient='h')
plt.axvline(ext_auc, color='red', linestyle='--', label='External Test')
plt.title('Internal CV vs External ROC-AUC')
plt.legend()
plt.show()